In [1]:
import pandas as pd
import sqlite3


In [7]:
# Load cleaned data
df = pd.read_csv("cleaned_data.csv")
df

,Segment,Country,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Date,Month Number,Month Name,Year
0,Government,Germany,Carretera,None,1513.0,3,350,529550.0,0.00,529550.00,393380.0,136170.00,2014-12-01,12,December,2014
1,Government,Germany,Paseo,None,1006.0,10,350,352100.0,0.00,352100.00,261560.0,90540.00,2014-06-01,6,June,2014
2,Government,Canada,Paseo,None,1725.0,10,350,603750.0,0.00,603750.00,448500.0,155250.00,2013-11-01,11,November,2013
3,Government,Germany,Paseo,None,1513.0,10,350,529550.0,0.00,529550.00,393380.0,136170.00,2014-12-01,12,December,2014
4,Government,Germany,Velo,None,1006.0,120,350,352100.0,0.00,352100.00,261560.0,90540.00,2014-06-01,6,June,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,Midmarket,Canada,Paseo,High,1614.0,10,15,24210.0,3631.50,20578.50,16140.0,4438.50,2014-04-01,4,April,2014
696,Midmarket,Canada,Paseo,High,2559.0,10,15,38385.0,5757.75,32627.25,25590.0,7037.25,2014-08-01,8,August,2014
697,Enterprise,Germany,Paseo,High,1085.0,10,125,135625.0,20343.75,115281.25,130200.0,-14918.75,2014-10-01,10,October,2014
698,Midmarket,Germany,Paseo,High,1175.0,10,15,17625.0,2643.75,14981.25,11750.0,3231.25,2014-10-01,10,October,2014


In [10]:
df['Date'] = pd.to_datetime(df['Date'])

In [11]:
print(df.dtypes)

Segment                        object
Country                        object
Product                        object
Discount Band                  object
Units Sold                    float64
Manufacturing Price             int64
Sale Price                      int64
Gross Sales                   float64
Discounts                     float64
Sales                         float64
COGS                          float64
Profit                        float64
Date                   datetime64[ns]
Month Number                    int64
Month Name                     object
Year                            int64
dtype: object


In [12]:
# === PART A: Pandas Analysis =====

print("=== Profit by Segment ===")
print(df.groupby('Segment')['Profit'].sum().sort_values(ascending=False))

print("\n=== Profit by Product ===")
print(df.groupby('Product')['Profit'].sum().sort_values(ascending=False))

print("\n=== Sales by Country ===")
print(df.groupby('Country')['Sales'].sum().sort_values(ascending=False))

print("\n=== Avg Profit by Discount Band ===")
print(df.groupby('Discount Band')['Profit'].mean().sort_values(ascending=False))

=== Profit by Segment ===
Segment
Government          1.138817e+07
Small Business      4.143168e+06
Channel Partners    1.316803e+06
Midmarket           6.601031e+05
Enterprise         -6.145456e+05
Name: Profit, dtype: float64

=== Profit by Product ===
Product
Paseo        4797437.950
VTT          3034608.020
Amarilla     2814104.060
Velo         2305992.465
Montana      2114754.880
Carretera    1826804.885
Name: Profit, dtype: float64

=== Sales by Country ===
Country
United States of America    2.502983e+07
Canada                      2.488765e+07
France                      2.435417e+07
Germany                     2.350534e+07
Mexico                      2.094935e+07
Name: Sales, dtype: float64

=== Avg Profit by Discount Band ===
Discount Band
Low       38680.360625
None      32763.301887
Medium    23055.879483
High      13832.109082
Name: Profit, dtype: float64


In [14]:
# === PART B: SQL Analysis (sqlite3) =====

conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')



In [15]:
print("\n=== SQL: Profit by Segment ===")
query1 = """
SELECT Segment, SUM(Profit) as Total_Profit
FROM sales
GROUP BY Segment
ORDER BY Total_Profit DESC
"""
print(pd.read_sql(query1, conn))



=== SQL: Profit by Segment ===
            Segment  Total_Profit
0        Government  1.138817e+07
1    Small Business  4.143168e+06
2  Channel Partners  1.316803e+06
3         Midmarket  6.601031e+05
4        Enterprise -6.145456e+05


In [16]:
print("\n=== SQL: Profit by Product ===")
query2 = """
SELECT Product, SUM(Profit) as Total_Profit
FROM sales
GROUP BY Product
ORDER BY Total_Profit DESC
"""
print(pd.read_sql(query2, conn))


=== SQL: Profit by Product ===
     Product  Total_Profit
0      Paseo   4797437.950
1        VTT   3034608.020
2   Amarilla   2814104.060
3       Velo   2305992.465
4    Montana   2114754.880
5  Carretera   1826804.885


In [17]:
print("\n=== SQL: Sales by Country ===")
query3 = """
SELECT Country, SUM(Sales) as Total_Sales
FROM sales
GROUP BY Country
ORDER BY Total_Sales DESC
"""
print(pd.read_sql(query3, conn))


=== SQL: Sales by Country ===
                    Country   Total_Sales
0  United States of America  2.502983e+07
1                    Canada  2.488765e+07
2                    France  2.435417e+07
3                   Germany  2.350534e+07
4                    Mexico  2.094935e+07


In [18]:
print("\n=== SQL: Top 5 Profitable Country+Product Combos ===")
query4 = """
SELECT Country, Product, SUM(Profit) as Total_Profit
FROM sales
GROUP BY Country, Product
ORDER BY Total_Profit DESC
LIMIT 5
"""
print(pd.read_sql(query4, conn))


=== SQL: Top 5 Profitable Country+Product Combos ===
                    Country Product  Total_Profit
0                    Canada   Paseo    1265017.99
1  United States of America   Paseo    1020603.27
2                    Mexico   Paseo     928651.39
3                    France   Paseo     838748.56
4                   Germany    Velo     788789.00
